# Mn-DOBDC Coordination Complex
## Electronic Properties and Metal–Ligand Binding Analysis

---

### What is this notebook about?

This notebook sits at the **middle level** of a three-notebook series on MOF-74-Mn:

| Notebook | System | Key questions |
|---|---|---|
| 1 · `dobdc_ligand_properties.ipynb` | Isolated H₄DOBDC ligand | Intrinsic organic HOMO/LUMO |
| **2 · This notebook** | **Mn-DOBDC coordination complex** | **Metal–ligand bonding, spin state, LMCT** |
| 3 · `mof74_mn_bulk.ipynb` | Bulk periodic MOF-74-Mn | Band gap, pores, adsorption |

---

### What is a Coordination Complex?

In MOF-74-Mn, each **Mn²⁺ metal node** is surrounded by oxygen atoms donated by
adjacent DOBDC linkers. A *coordination complex* is a molecular model that captures
that local bonding environment — close enough to the real MOF environment to answer
electronic-structure questions without the cost of a full periodic calculation.

**MOF-74 binding site geometry:**
- Each Mn²⁺ is *octahedrally* coordinated by **5 oxygen donors from DOBDC** and
  **1 open metal site** (a water molecule that desorbs on activation).
- The DOBDC ligand binds in a **bidentate** fashion through two adjacent oxygens:
  - *O₁*: the **carboxylate** oxygen (from –COO⁻)
  - *O₂*: the **phenolate** oxygen (from –O⁻ on the aromatic ring)
  - Together they form a **6-membered chelate ring**: Mn–O₁–C(carboxyl)–C(ring)–C(ring)–O₂–(back to Mn)

---

### Why does Mn²⁺ spin state matter?

Mn²⁺ is a d⁵ ion. In the *weak-field* oxygen donor environment of MOF-74,
it adopts the **high-spin** configuration: all five d electrons are unpaired.

```
d-orbital filling (high-spin d⁵):  ↑  ↑  ↑  ↑  ↑
                                   dxy dxz dyz dx² dz²

Unpaired electrons (UHF):  5
Spin multiplicity (2S+1):  6   ← sextet state
```

This is important because:
1. Any DFT/semi-empirical method must be run **unrestricted** (UHF/UKS).
2. The singly-occupied d orbitals couple with the DOBDC π system → **LMCT** bands.
3. Magnetic properties of MOF-74-Mn arise from exchange coupling between these spins.

---

### Model used here

`[Mn(DOBDC)(H₂O)₄]²⁻`

```
Mn²⁺  (+2)  +  DOBDC⁴⁻ (–4)  +  4 × H₂O (0)  =  charge –2
                                                     UHF  = 5 (high-spin d⁵)
```

This represents one Mn²⁺ coordinated at a single bidentate DOBDC arm, with
four water molecules completing octahedral coordination — a standard first-
principles cluster model for MOF-74 metal nodes.


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
SMILES_DOBDC    = "OC(=O)c1cc(O)c(C(=O)O)cc1O"  # neutral H₄DOBDC for geometry
COMPLEX_CHARGE  = -2   # Mn²⁺ (+2) + DOBDC⁴⁻ (−4) + 4×H₂O (0) = −2
MN_SPIN         = 5    # number of unpaired electrons (UHF parameter)
MN_MULTIPLICITY = 6    # 2S+1 = 2×(5/2)+1 = 6  (sextet)
CUTOFF          = 5.0  # AtomicGraph neighbour-list cutoff (Å)
MN_O_CHELATE    = 2.10 # Mn–O bond to chelate oxygens (Å, from crystal data)
MN_O_WATER      = 2.17 # Mn–O(water) bond length (Å)
MACE_MODEL      = "small"
ORCA_CMD        = None  # set to "/absolute/path/to/orca" if available
WORK_DIR        = "dobdc_work"


In [ ]:
import os, re, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ase import Atoms
from ase.optimize import BFGS
from ase.neighborlist import NeighborList
from goal.ml.data.graph import AtomicGraph

os.makedirs(WORK_DIR, exist_ok=True)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    HAS_RDKIT = True
except ImportError:
    HAS_RDKIT = False; print("RDKit missing — geometry generation unavailable")

try:
    import py3Dmol; HAS_PY3DMOL = True
except ImportError:
    HAS_PY3DMOL = False

try:
    from xtb.ase.calculator import XTB as XTBCalcASE; HAS_XTB_ASE = True
except ImportError:
    HAS_XTB_ASE = False

try:
    from xtb.interface import Calculator as XTBInterface, Param; HAS_XTB_IFACE = True
except ImportError:
    HAS_XTB_IFACE = False

try:
    from mace.calculators import mace_mp; HAS_MACE = True
except ImportError:
    HAS_MACE = False; print("MACE missing — pip install mace-torch")

try:
    from ase.calculators.orca import ORCA, OrcaProfile, OrcaTemplate; HAS_ORCA = True
except ImportError:
    HAS_ORCA = False

# ── CPK colours (extend with common metal colours) ──────────────────────────
CPK = {
    "H":"#FFFFFF","C":"#808080","N":"#4169E1","O":"#FF4500","S":"#FFD700",
    "F":"#9ACD32","Cl":"#00FF00","Br":"#8B0000","I":"#940094","P":"#FFA500",
    "Mn":"#9C3E8F","Fe":"#E06000","Co":"#0000C8","Ni":"#28A028",
    "Cu":"#C88033","Zn":"#7D80B0","Mg":"#8AFF00","Ca":"#3DFF00",
}
VDW = {  # CPK radii (display scale ×0.35)
    "H":0.31,"C":0.77,"N":0.75,"O":0.73,"S":1.02,"Mn":1.61,"Fe":1.56,
    "Co":1.52,"Ni":1.49,"Cu":1.45,"Zn":1.42,"Mg":1.60,"Ca":1.74,
}
print("Imports done.")


## Section 2 · Building the Mn-DOBDC Coordination Complex

In [ ]:
if not HAS_RDKIT:
    raise RuntimeError("RDKit required for geometry generation")

# ── Step 1: Build neutral H₄DOBDC (MMFF94 for stable 3-D geometry) ──────────
mol  = Chem.MolFromSmiles(SMILES_DOBDC)
mol  = Chem.AddHs(mol)
ps   = AllChem.ETKDGv3(); ps.randomSeed = 42
AllChem.EmbedMolecule(mol, ps)
AllChem.MMFFOptimizeMolecule(mol, mmffVariant="MMFF94s")

conf    = mol.GetConformer(0)
all_pos = np.array(conf.GetPositions())   # (N, 3) in Å
all_sym = [a.GetSymbol() for a in mol.GetAtoms()]

print(f"H₄DOBDC built: {len(all_sym)} atoms "
      f"({sum(s != 'H' for s in all_sym)} heavy + {sum(s == 'H' for s in all_sym)} H)")


In [ ]:
# ── Step 2: Identify bidentate O atoms and ionizable H ───────────────────────
#
# Bidentate pair = one C=O oxygen (carboxylate) + one Ar-OH oxygen (phenolate)
# from the same end of DOBDC.  They will be ~2.8–3.1 Å apart in the 6-membered
# chelate ring: Mn–O₁–C(carboxyl)–C₁(ring)–C₂(ring)–O₂–Mn.

carbonyl_o_idx = []   # C=O oxygens (one per COOH, double bond to non-aromatic C)
phenol_o_idx   = []   # Ar-OH oxygens (single bond to aromatic C, has H)
ionizable_h    = set()  # H atoms to remove when deprotonating to DOBDC⁴⁻

for atom in mol.GetAtoms():
    anum = atom.GetAtomicNum()
    idx  = atom.GetIdx()
    nbrs = list(atom.GetNeighbors())

    if anum == 8:   # oxygen
        has_arom_C = any(n.GetAtomicNum() == 6 and n.GetIsAromatic() for n in nbrs)
        has_H      = any(n.GetAtomicNum() == 1 for n in nbrs)
        has_dbl_C  = any(
            mol.GetBondBetweenAtoms(idx, n.GetIdx()).GetBondTypeAsDouble() == 2.0
            and n.GetAtomicNum() == 6 and not n.GetIsAromatic()
            for n in nbrs
        )
        if has_dbl_C and not has_H:   # C=O, no H → carbonyl oxygen
            carbonyl_o_idx.append(idx)
        if has_arom_C and has_H:      # Ar-OH → phenolate candidate
            phenol_o_idx.append(idx)

    if anum == 1:   # hydrogen
        if len(nbrs) == 1 and nbrs[0].GetAtomicNum() == 8:   # H bonded to O
            o = nbrs[0]
            o_nbrs = list(o.GetNeighbors())
            is_phenol_h  = any(n.GetAtomicNum() == 6 and n.GetIsAromatic() for n in o_nbrs)
            is_cooh_h    = any(
                n.GetAtomicNum() == 6 and not n.GetIsAromatic() and
                any(mol.GetBondBetweenAtoms(n.GetIdx(), nn.GetIdx()).GetBondTypeAsDouble() == 2.0
                    and nn.GetAtomicNum() == 8 for nn in n.GetNeighbors() if nn != o)
                for n in o_nbrs
            )
            if is_phenol_h or is_cooh_h:
                ionizable_h.add(idx)

print(f"Carbonyl O (C=O):   {carbonyl_o_idx}")
print(f"Phenolate O (Ar-OH): {phenol_o_idx}")
print(f"Ionizable H:         {sorted(ionizable_h)}")

# Find bidentate pairs: carbonyl_O + phenol_O within 3.5 Å  (6-membered chelate)
bidentate = []
for ci in carbonyl_o_idx:
    for pi in phenol_o_idx:
        d = np.linalg.norm(all_pos[ci] - all_pos[pi])
        if d < 3.5:
            bidentate.append((ci, pi, d))
            print(f"  Bidentate pair: O{ci}(C=O) – O{pi}(Ar-OH)  d = {d:.3f} Å")

assert bidentate, "No bidentate pair found — check SMILES"
print(f"\nUsing site A (pair 0) for Mn placement.")


In [ ]:
# ── Step 3: Place Mn²⁺ at bidentate site ─────────────────────────────────────
#
# Strategy: Mn sits on the perpendicular bisector of the O1–O2 axis, displaced
# outward from the ring centre so that Mn–O1 ≈ Mn–O2 ≈ MN_O_CHELATE.
#
#   d_perp = sqrt(MN_O_CHELATE² − (d_OO/2)²)

ci, pi, d_oo = bidentate[0]
o1 = all_pos[ci]   # carboxylate O
o2 = all_pos[pi]   # phenolate O

mid     = (o1 + o2) / 2
oo_unit = (o2 - o1) / np.linalg.norm(o2 - o1)

# Outward direction: from ring centroid through midpoint, projected ⊥ to O–O axis
ring_c   = np.array([all_pos[i] for i, s in enumerate(all_sym) if s == 'C']).mean(0)
raw_out  = mid - ring_c
out_perp = raw_out - np.dot(raw_out, oo_unit) * oo_unit
if np.linalg.norm(out_perp) < 1e-6:
    out_perp = np.array([0., 0., 1.])
    out_perp -= np.dot(out_perp, oo_unit) * oo_unit
out_perp /= np.linalg.norm(out_perp)

d_perp = np.sqrt(max(0.0, MN_O_CHELATE**2 - (d_oo / 2)**2))
mn_pos = mid + d_perp * out_perp

# Verify distances
d1 = np.linalg.norm(mn_pos - o1)
d2 = np.linalg.norm(mn_pos - o2)
ang = np.degrees(np.arccos(np.clip(np.dot(o1-mn_pos, o2-mn_pos) / (d1*d2), -1, 1)))
print(f"Mn–O₁ (carboxylate): {d1:.3f} Å  (target {MN_O_CHELATE} Å)")
print(f"Mn–O₂ (phenolate):   {d2:.3f} Å  (target {MN_O_CHELATE} Å)")
print(f"O₁–Mn–O₂ angle:      {ang:.1f}°  (ideal chelate ~65°)")


In [ ]:
# ── Step 4: Add 4 water ligands (complete octahedral coordination) ────────────
#
# Coordinate frame at Mn:
#   v1, v2   = unit vectors from Mn to O1, O2
#   n        = v1 × v2  (normal to O1–Mn–O2 plane)
#   inplane  = n × bisect  (in-plane ⊥ to bisector)

v1      = (o1 - mn_pos); v1 /= np.linalg.norm(v1)
v2      = (o2 - mn_pos); v2 /= np.linalg.norm(v2)
n       = np.cross(v1, v2)
if np.linalg.norm(n) > 1e-6:
    n /= np.linalg.norm(n)
else:
    n = np.array([0., 0., 1.])
bisect   = (v1 + v2); bisect /= np.linalg.norm(bisect)
inplane  = np.cross(n, bisect); inplane /= np.linalg.norm(inplane)

water_o_dirs = [n, -n, inplane, -inplane]   # 4 octahedral sites

w_syms, w_pos = [], []
for wd in water_o_dirs:
    ow = mn_pos + MN_O_WATER * wd            # water O

    # H atoms: H–O–H angle 104.5°, O–H = 0.958 Å, H atoms point away from Mn
    perp = np.array([1., 0., 0.])
    perp -= np.dot(perp, wd) * wd
    if np.linalg.norm(perp) < 1e-6:
        perp = np.array([0., 1., 0.])
        perp -= np.dot(perp, wd) * wd
    perp /= np.linalg.norm(perp)

    half = np.radians(52.25)               # 104.5° / 2
    oh   = 0.958
    h1   = ow + oh * (np.cos(half)*(-wd) + np.sin(half)*perp)
    h2   = ow + oh * (np.cos(half)*(-wd) - np.sin(half)*perp)

    w_syms += ["O","H","H"]
    w_pos  += [ow, h1, h2]

w_pos = np.array(w_pos)

# ── Step 5: Assemble DOBDC⁴⁻ + Mn²⁺ + 4 H₂O ─────────────────────────────────
keep = [i not in ionizable_h for i in range(len(all_sym))]
d4_sym  = [s for i, s in enumerate(all_sym) if keep[i]]
d4_pos  = all_pos[np.array(keep)]

dobdc4  = Atoms(d4_sym,  positions=d4_pos)
mn_atom = Atoms(["Mn"],  positions=[mn_pos])
waters  = Atoms(w_syms,  positions=w_pos)

atoms = dobdc4 + mn_atom + waters
atoms.write(f"{WORK_DIR}/mn_dobdc_complex_initial.xyz")

print(f"Complex assembled: {len(atoms)} atoms")
print(f"  DOBDC⁴⁻ : {len(dobdc4)} atoms (C₈H₂O₆)")
print(f"  Mn²⁺    : 1 atom")
print(f"  4×H₂O   : {len(waters)} atoms")
print(f"  Charge  : {COMPLEX_CHARGE}  (Mn²⁺ + DOBDC⁴⁻ + 4H₂O)")
print(f"  UHF (↑−↓) : {MN_SPIN}  (high-spin d⁵)")
print(f"  Multiplicity: {MN_MULTIPLICITY}")
print(f"Saved → {WORK_DIR}/mn_dobdc_complex_initial.xyz")


### 2.1 · 3D Structure Preview — [Mn(DOBDC)(H₂O)₄]²⁻

In [ ]:
if not HAS_PY3DMOL:
    print("py3Dmol not available"); raise SystemExit

_syms = atoms.get_chemical_symbols()
_pos  = atoms.get_positions()
_nat  = len(_syms)

# Build neighbour list for bonds
_rcuts = [{"H":0.31,"C":0.77,"N":0.75,"O":0.73,"Mn":1.45}.get(s,1.0)
          for s in _syms]
_nl = NeighborList(_rcuts, skin=0.2, self_interaction=False, bothways=False)
_nl.update(atoms)

view = py3Dmol.view(width=720, height=500)
view.setBackgroundColor("#1a1a2e")

# Atoms
for i, (sym, pos) in enumerate(zip(_syms, _pos)):
    col = CPK.get(sym, "#AAAAAA")
    rad = VDW.get(sym, 0.8) * (0.45 if sym == "Mn" else 0.30)
    view.addSphere({"center":{"x":float(pos[0]),"y":float(pos[1]),"z":float(pos[2])},
                    "radius":rad, "color":col, "opacity":1.0})

# Bonds
for i in range(_nat):
    idxs, _ = _nl.get_neighbors(i)
    for j in idxs:
        p1, p2 = _pos[i], _pos[j]
        mid3 = (p1 + p2) / 2
        c1, c2 = CPK.get(_syms[i],"#AAAAAA"), CPK.get(_syms[j],"#AAAAAA")
        for (st, en, col) in [(p1, mid3, c1), (mid3, p2, c2)]:
            view.addCylinder({"start":{"x":float(st[0]),"y":float(st[1]),"z":float(st[2])},
                              "end":  {"x":float(en[0]),"y":float(en[1]),"z":float(en[2])},
                              "radius":0.07, "color":col, "opacity":1.0})

# Atom labels for Mn and O donors
for i, (sym, pos) in enumerate(zip(_syms, _pos)):
    if sym == "Mn" or (sym == "O" and i < len(dobdc4) + 1):
        view.addLabel(sym, {"position":{"x":float(pos[0]),"y":float(pos[1]),"z":float(pos[2])},
                            "fontColor":"white", "backgroundColor":"transparent",
                            "fontSize":12, "fontOpacity":0.9})

view.zoomTo()
view.show()


## Section 3 · AtomicGraph

In [ ]:
graph = AtomicGraph.from_ase(atoms, cutoff=CUTOFF)

print("AtomicGraph  [Mn(DOBDC)(H₂O)₄]²⁻")
print(f"  Nodes (atoms)   : {graph.num_nodes}")
print(f"  Edges (pairs)   : {graph.num_edges}")
print(f"  pos  shape      : {graph.positions.shape}")
print(f"  z    shape      : {graph.z.shape}")
print(f"  edge_index shape: {graph.edge_index.shape}")
print(f"  PBC active      : {graph.pbc.any().item()}")

from collections import Counter
elem_count = Counter(atoms.get_chemical_symbols())
print(f"\nElement composition: {dict(elem_count)}")


## Section 4 · MACE-MP-0 Force Field

In [ ]:
if not HAS_MACE:
    raise RuntimeError("Install MACE: pip install mace-torch")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
mace_calc = mace_mp(model=MACE_MODEL, dispersion=False, device=device,
                    default_dtype="float32")
print(f"MACE-MP-0 ({MACE_MODEL}) loaded on {device}")


In [ ]:
atoms_mace = atoms.copy()
atoms_mace.calc = mace_calc
e_mace = atoms_mace.get_potential_energy()
f_mace = atoms_mace.get_forces()

print(f"MACE energy  : {e_mace:.4f} eV")
print(f"Force max    : {np.abs(f_mace).max():.4f} eV/Å")
print(f"Force RMS    : {np.sqrt((f_mace**2).mean()):.4f} eV/Å")

# Force overview
syms = atoms.get_chemical_symbols()
print("\n  Atom  |  |F| (eV/Å)")
for i, (s, f) in enumerate(zip(syms, f_mace)):
    if s == "Mn" or (s in ("O","C") and np.linalg.norm(f) > 0.3):
        print(f"  {i:3d} {s:2s}  |  {np.linalg.norm(f):.3f}")


In [ ]:
print("Running BFGS geometry optimisation with MACE …")
atoms_opt = atoms.copy()
atoms_opt.calc = mace_calc
opt = BFGS(atoms_opt, logfile=f"{WORK_DIR}/mn_dobdc_bfgs.log")
opt.run(fmax=0.05, steps=500)
atoms_opt.write(f"{WORK_DIR}/mn_dobdc_complex_mace.xyz")

# Check Mn coordination after relaxation
mn_i = [i for i, s in enumerate(atoms_opt.get_chemical_symbols()) if s == "Mn"][0]
mn_p = atoms_opt.get_positions()[mn_i]
print(f"\nMn position after relaxation: {mn_p}")
print("Mn–X bond distances:")
for i, (s, p) in enumerate(zip(atoms_opt.get_chemical_symbols(),
                                atoms_opt.get_positions())):
    d = np.linalg.norm(p - mn_p)
    if i != mn_i and d < 2.5:
        print(f"  Mn–{s}{i}: {d:.3f} Å")

print(f"\nSaved → {WORK_DIR}/mn_dobdc_complex_mace.xyz")
atoms_qm = atoms_opt.copy()   # use the MACE-relaxed geometry for QM


## Section 5 · GFN2-xTB Electronic Properties

GFN2-xTB is a semi-empirical tight-binding method that correctly handles:
- Open-shell (UHF) calculations via the `uhf=N` parameter
- Transition metal d-electrons including Mn
- Orbital eigenvalues → HOMO, LUMO, gap

**Settings for this complex:**
```
charge = -2   # Mn²⁺(+2) + DOBDC⁴⁻(−4) + 4H₂O(0)
uhf    = 5    # 5 unpaired electrons (high-spin Mn²⁺ d⁵)
```


In [ ]:
if not HAS_XTB_ASE:
    print("xtb-python ASE calculator not available"); raise SystemExit

atoms_xtb = atoms_qm.copy()
calc_xtb  = XTBCalcASE(method="GFN2-xTB", charge=COMPLEX_CHARGE, uhf=MN_SPIN)
atoms_xtb.calc = calc_xtb

e_xtb = atoms_xtb.get_potential_energy()
f_xtb = atoms_xtb.get_forces()
print(f"GFN2-xTB single-point complete")
print(f"  Energy : {e_xtb:.4f} eV")
print(f"  |F|max : {np.abs(f_xtb).max():.4f} eV/Å")


In [ ]:
if not HAS_XTB_IFACE:
    print("xtb.interface not available — cannot extract orbital eigenvalues"); raise SystemExit

import ase.units as units

pos_bohr = atoms_qm.get_positions() / units.Bohr
numbers  = atoms_qm.get_atomic_numbers()

calc_iface = XTBInterface(Param.GFN2xTB, numbers, pos_bohr,
                          charge=float(COMPLEX_CHARGE), uhf=MN_SPIN)
calc_iface.singlepoint()

ev_per_ha = units.Hartree          # conversion factor
orb_eigs  = calc_iface.get_orbital_eigenvalues()   # shape (2, n_orb) for UHF, or (n_orb,)
dipole    = calc_iface.get_dipole()

if orb_eigs.ndim == 2:
    alpha_eigs = orb_eigs[0] * ev_per_ha   # alpha (↑) spin channel
    beta_eigs  = orb_eigs[1] * ev_per_ha   # beta  (↓) spin channel
    # HOMO of alpha channel (highest occupied), LUMO of beta channel
    n_alpha = (numbers.sum() + MN_SPIN) // 2
    n_beta  = (numbers.sum() - MN_SPIN) // 2
    homo_eV = alpha_eigs[n_alpha - 1]
    lumo_eV = beta_eigs[n_beta]             # lowest unoccupied in beta channel
else:
    eigs     = orb_eigs * ev_per_ha
    n_occ    = int(numbers.sum() - COMPLEX_CHARGE) // 2
    homo_eV  = eigs[n_occ - 1]
    lumo_eV  = eigs[n_occ]

gap_eV = lumo_eV - homo_eV

print("── GFN2-xTB Orbital Energies ──────────────────────────")
print(f"  HOMO  : {homo_eV:+.4f} eV")
print(f"  LUMO  : {lumo_eV:+.4f} eV")
print(f"  Gap   : {gap_eV:.4f} eV")
print(f"  Dipole: {np.linalg.norm(dipole):.3f} Debye")


In [ ]:
# ── Frontier MO diagram (α channel) ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 7))
ax.set_facecolor("#1a1a2e"); fig.patch.set_facecolor("#1a1a2e")
for sp in ax.spines.values(): sp.set_color("#AAAAAA")
ax.tick_params(colors="#CCCCCC"); ax.yaxis.label.set_color("#CCCCCC")
ax.set_title("Frontier MOs — [Mn(DOBDC)(H₂O)₄]²⁻\nGFN2-xTB (α channel)",
             color="#CCCCCC", fontsize=11)

# Show orbitals within ±3 eV of HOMO/LUMO
eigs = alpha_eigs if orb_eigs.ndim == 2 else (orb_eigs * units.Hartree)
window = [e for e in eigs if homo_eV - 3 < e < lumo_eV + 3]

for e in window:
    occ = "occ" if e <= homo_eV else "virt"
    col = "#50C878" if occ == "occ" else "#FF6B6B"
    ax.axhline(e, xmin=0.2, xmax=0.8, color=col, lw=2)

ax.axhline(homo_eV, xmin=0.15, xmax=0.85, color="#FFD700", lw=3, label=f"HOMO {homo_eV:.3f} eV")
ax.axhline(lumo_eV, xmin=0.15, xmax=0.85, color="#FF6B6B", lw=3, label=f"LUMO {lumo_eV:.3f} eV")
ax.annotate("", xy=(0.5, lumo_eV), xytext=(0.5, homo_eV),
            xycoords=("axes fraction","data"), textcoords=("axes fraction","data"),
            arrowprops=dict(arrowstyle="<->", color="white", lw=1.5))
ax.text(0.55, (homo_eV+lumo_eV)/2, f"Δ = {gap_eV:.2f} eV",
        transform=ax.get_yaxis_transform(), color="white", va="center", fontsize=10)

ax.set_ylabel("Energy (eV)", color="#CCCCCC")
ax.set_xticks([]); ax.legend(facecolor="#2a2a4e", labelcolor="white")
plt.tight_layout(); plt.show()
print(f"HOMO-LUMO gap: {gap_eV:.3f} eV")


In [ ]:
# ── Mulliken charges ──────────────────────────────────────────────────────────
charges = calc_iface.get_partial_charges()
syms_qm = atoms_qm.get_chemical_symbols()

print("Selected Mulliken charges (GFN2-xTB):")
print(f"  Mn  charge: {charges[[i for i,s in enumerate(syms_qm) if s=='Mn'][0]]:+.3f} e")

fig, ax = plt.subplots(figsize=(10, 3))
ax.set_facecolor("#1a1a2e"); fig.patch.set_facecolor("#1a1a2e")
for sp in ax.spines.values(): sp.set_color("#AAAAAA")
ax.tick_params(colors="#CCCCCC")
ax.set_title("Mulliken Charges — [Mn(DOBDC)(H₂O)₄]²⁻", color="#CCCCCC")

x = np.arange(len(charges))
cols = ["#FF6B6B" if q > 0 else "#50C878" for q in charges]
ax.bar(x, charges, color=cols, alpha=0.8)
ax.axhline(0, color="white", lw=0.8)
ax.set_xlabel("Atom index", color="#CCCCCC")
ax.set_ylabel("Charge (e)", color="#CCCCCC")
plt.tight_layout(); plt.show()


## Section 6 · ORCA DFT Calculation

For transition metal complexes, ORCA with a **spin-unrestricted** functional is recommended:

| Method | Notes |
|---|---|
| `BP86 def2-SVP` | Good for geometry/frequency; fast |
| `B3LYP def2-TZVP` | More accurate energetics; ~2× slower |
| `TPSS def2-SVP` | Good balance for transition metals |
| `!XTB2` | ORCA's built-in xTB (very fast, same results as Section 5) |

> **Spin state**: ORCA will automatically run UKS when `multiplicity > 1`.
> With `multiplicity=6` (sextet), ORCA sets up α/β spin-orbital treatment — no
> extra keywords needed.

**Charge**: `−2` (set in `COMPLEX_CHARGE`)
**Multiplicity**: `6` (set in `MN_MULTIPLICITY`)


In [ ]:
def parse_orca_orbital_energies(orca_out):
    """Extract orbital energies (eV) from an ORCA output file."""
    text = pathlib.Path(orca_out).read_text()
    energies, spins = [], []
    for block_match in re.finditer(
            r'ORBITAL ENERGIES.*?(?=\n\n|\Z)', text, re.DOTALL):
        block = block_match.group()
        for line in block.splitlines():
            parts = line.split()
            if len(parts) >= 4:
                try:
                    occ  = float(parts[1])
                    e_ha = float(parts[2])
                    e_ev = float(parts[3])
                    energies.append((occ, e_ev))
                except ValueError:
                    continue
    if not energies:
        raise ValueError(f"No orbital energies found in {orca_out}")
    occs = np.array([e[0] for e in energies])
    eigs = np.array([e[1] for e in energies])
    # HOMO = highest occupied, LUMO = lowest unoccupied
    occ_mask = occs > 0.5
    homo_ev = eigs[occ_mask][-1]
    lumo_ev = eigs[~occ_mask][0]
    return eigs, occ_mask, homo_ev, lumo_ev

import pathlib
print("ORCA parsing helper defined.")


In [ ]:
if not HAS_ORCA or ORCA_CMD is None:
    print("ORCA not configured — skipping (set ORCA_CMD = '/path/to/orca')")
    homo_orca, lumo_orca = homo_eV, lumo_eV   # fall back to xTB values
else:
    orca_dir = f"{WORK_DIR}/orca_complex"
    os.makedirs(orca_dir, exist_ok=True)

    # BP86/def2-SVP is a reliable, fast choice for Mn complexes
    kw = ["BP86", "def2-SVP", "D3BJ", "RIJCOSX", "def2/J", "EnGrad"]

    calc_orca = ORCA(
        profile=OrcaProfile(command=ORCA_CMD),
        template=OrcaTemplate(
            keywords=" ".join(kw),
            charge=COMPLEX_CHARGE,
            multiplicity=MN_MULTIPLICITY,
        ),
        directory=orca_dir,
    )

    atoms_orca = atoms_qm.copy()
    atoms_orca.calc = calc_orca
    e_orca = atoms_orca.get_potential_energy()
    f_orca = atoms_orca.get_forces()

    orca_out = list(pathlib.Path(orca_dir).glob("*.out"))[0]
    eigs_orca, occ_mask_orca, homo_orca, lumo_orca = parse_orca_orbital_energies(str(orca_out))
    gap_orca = lumo_orca - homo_orca

    print(f"ORCA  BP86/def2-SVP  (charge={COMPLEX_CHARGE}, mult={MN_MULTIPLICITY})")
    print(f"  Energy : {e_orca:.4f} eV")
    print(f"  HOMO   : {homo_orca:+.4f} eV")
    print(f"  LUMO   : {lumo_orca:+.4f} eV")
    print(f"  Gap    : {gap_orca:.4f} eV")


## Section 7 · Binding Energy Analysis

In [ ]:
# ── Binding energy via GFN2-xTB ───────────────────────────────────────────────
#
# ΔE_bind = E(complex) − E(Mn²⁺) − E(DOBDC⁴⁻) − 4×E(H₂O)
#
# Each fragment uses its own charge and spin:
#   Mn²⁺ alone   : charge=+2, uhf=5  (isolated d⁵ ion)
#   DOBDC⁴⁻      : charge=-4, uhf=0  (closed-shell anion)
#   H₂O          : charge= 0, uhf=0
#   Complex       : charge=-2, uhf=5  (already computed above)

if not HAS_XTB_ASE:
    print("xTB not available — cannot compute binding energy"); raise SystemExit

# Mn²⁺ alone (just the metal ion)
mn_only = Atoms(["Mn"], positions=[[0,0,0]])
mn_only.calc = XTBCalcASE(method="GFN2-xTB", charge=2, uhf=MN_SPIN)
e_mn = mn_only.get_potential_energy()

# DOBDC⁴⁻ alone (optimised geometry)
dobdc_atoms = Atoms(d4_sym, positions=d4_pos)
dobdc_atoms.calc = XTBCalcASE(method="GFN2-xTB", charge=-4, uhf=0)
e_dobdc = dobdc_atoms.get_potential_energy()

# One water molecule
from ase.build import molecule as ase_mol
h2o = ase_mol("H2O")
h2o.calc = XTBCalcASE(method="GFN2-xTB", charge=0, uhf=0)
e_water = h2o.get_potential_energy()

# Complex energy
e_complex = e_xtb   # from Section 5

e_bind = e_complex - e_mn - e_dobdc - 4*e_water
print("── Binding Energy (GFN2-xTB) ─────────────────────────────")
print(f"  E(complex)   = {e_complex:.4f} eV")
print(f"  E(Mn²⁺)      = {e_mn:.4f} eV")
print(f"  E(DOBDC⁴⁻)   = {e_dobdc:.4f} eV")
print(f"  E(H₂O)×4     = {4*e_water:.4f} eV")
print(f"  ─────────────────────────────────────────────")
print(f"  ΔE_bind      = {e_bind:.4f} eV  ({e_bind*23.06:.1f} kcal/mol)")
print()
if e_bind < 0:
    print("  → Negative binding energy: complex is thermodynamically STABLE ✓")
else:
    print("  → Positive binding energy: GFN2-xTB may not favour this charge state")
    print("    Consider using ORCA with a proper DFT functional for quantitative results.")


## Section 8 · Conceptual DFT Descriptors

In [ ]:
# ── Conceptual DFT (Parr–Pearson hardness / softness) ─────────────────────────
#
#  IP  = −HOMO   (ionisation potential)
#  EA  = −LUMO   (electron affinity)
#  η   = (IP − EA)/2 = (gap)/2       (chemical hardness)
#  μ   = (HOMO + LUMO)/2              (chemical potential = −electronegativity)
#  χ   = −μ                           (electronegativity)
#  ω   = μ²/(2η)                      (global electrophilicity index)

homo = homo_eV;  lumo = lumo_eV

IP  = -homo
EA  = -lumo
eta = (IP - EA) / 2
mu  = (homo + lumo) / 2
chi = -mu
gap = lumo - homo
S   = 1 / (2*eta) if eta != 0 else float("inf")
omega = mu**2 / (2*eta) if eta != 0 else float("inf")

print("── Conceptual DFT Descriptors ─────────────────────────────")
print(f"  HOMO  : {homo:+.3f} eV")
print(f"  LUMO  : {lumo:+.3f} eV")
print(f"  Gap   : {gap:.3f} eV")
print(f"  IP    : {IP:.3f} eV")
print(f"  EA    : {EA:.3f} eV")
print(f"  η (hardness)       : {eta:.3f} eV")
print(f"  S (softness)       : {S:.3f} eV⁻¹")
print(f"  μ (chem. potential): {mu:+.3f} eV")
print(f"  χ (electroneg.)    : {chi:.3f} eV")
print(f"  ω (electrophilicity): {omega:.3f} eV")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("#1a1a2e")

labels   = ["IP", "EA", "η", "μ", "χ", "ω"]
values   = [IP, EA, eta, mu, chi, omega]
units_l  = ["eV","eV","eV","eV","eV","eV"]
colours  = ["#FF6B6B","#50C878","#4169E1","#FFD700","#FF69B4","#00BFFF"]

ax = axes[0]
ax.set_facecolor("#1a1a2e")
for sp in ax.spines.values(): sp.set_color("#AAAAAA")
ax.tick_params(colors="#CCCCCC")
ax.set_title("Conceptual DFT Descriptors\n[Mn(DOBDC)(H₂O)₄]²⁻", color="#CCCCCC")
bars = ax.bar(labels, values, color=colours, alpha=0.85, edgecolor="#333355")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02*abs(max(values)),
            f"{val:.2f}", ha="center", va="bottom", color="white", fontsize=9)
ax.axhline(0, color="white", lw=0.7)
ax.set_ylabel("eV", color="#CCCCCC")

# HOMO-LUMO level diagram
ax2 = axes[1]
ax2.set_facecolor("#1a1a2e")
for sp in ax2.spines.values(): sp.set_color("#AAAAAA")
ax2.tick_params(colors="#CCCCCC")
ax2.set_title("HOMO–LUMO Level Diagram", color="#CCCCCC")
ax2.axhline(homo, xmin=0.15, xmax=0.45, color="#FFD700", lw=3, label=f"HOMO {homo:.2f} eV")
ax2.axhline(lumo, xmin=0.55, xmax=0.85, color="#FF6B6B", lw=3, label=f"LUMO {lumo:.2f} eV")
ax2.annotate("", xy=(0.5, lumo), xytext=(0.5, homo),
             xycoords=("axes fraction","data"), textcoords=("axes fraction","data"),
             arrowprops=dict(arrowstyle="<->", color="white", lw=1.5))
ax2.text(0.52, (homo+lumo)/2, f"Δ = {gap:.2f} eV",
         transform=ax2.get_yaxis_transform(), color="white", va="center")
ax2.set_ylabel("Energy (eV)", color="#CCCCCC"); ax2.set_xticks([])
ax2.legend(facecolor="#2a2a4e", labelcolor="white")

plt.tight_layout(); plt.show()


## Summary

In [ ]:
print("═"*60)
print("  SUMMARY: [Mn(DOBDC)(H₂O)₄]²⁻  (GFN2-xTB)")
print("═"*60)
print(f"  Charge / UHF    : {COMPLEX_CHARGE} / {MN_SPIN} (sextet Mn²⁺ d⁵)")
print(f"  Atoms           : {len(atoms_qm)}")
print(f"  MACE energy     : {e_mace:.4f} eV")
print(f"  xTB  energy     : {e_xtb:.4f} eV")
print(f"  HOMO            : {homo_eV:+.4f} eV")
print(f"  LUMO            : {lumo_eV:+.4f} eV")
print(f"  HOMO-LUMO gap   : {gap_eV:.4f} eV")
print(f"  ΔE_bind (xTB)   : {e_bind:.4f} eV  ({e_bind*23.06:.1f} kcal/mol)")
print(f"  η  (hardness)   : {eta:.4f} eV")
print(f"  χ  (electroneg.): {chi:.4f} eV")
print(f"  ω  (electrophil): {omega:.4f} eV")
print()
import glob
for f in sorted(glob.glob(f"{WORK_DIR}/mn_dobdc*")):
    print(f"  {f}")
print("═"*60)
